# 🤖 Hermes Agent — Google Colab (Gemini)

| Item | Detail |
|------|--------|
| Provider | Google Gemini (`GEMINI_API_KEY`) |
| Model | `gemini/gemini-2.5-flash` |
| Binary | `/content/hermes-agent/.venv/bin/hermes` |
| Chat mode | `-z` (oneshot) |

> ⚠️ Colab gratis **tidak persisten** — instalasi & memori hilang saat runtime mati.
> Untuk 24/7, gunakan VPS.

---
**Jalankan sel berurutan dari atas ke bawah.**

## 1️⃣ Instalasi
*Jalankan sekali per runtime baru.*

In [ ]:
import os

if os.path.isdir('/content/hermes-agent'):
    print('✅ Repo sudah ada.')
else:
    print('⏳ Clone repo... (~345 MB)')
    !git clone https://github.com/NousResearch/hermes-agent.git /content/hermes-agent
    print('✅ Selesai.')

In [ ]:
%cd /content/hermes-agent

if os.path.isdir('.venv'):
    print('✅ venv sudah ada.')
else:
    print('⏳ Buat venv Python 3.11...')
    !/usr/local/bin/uv venv .venv --python 3.11
    print('✅ Selesai.')

In [ ]:
%cd /content/hermes-agent

HERMES = '/content/hermes-agent/.venv/bin/hermes'

if os.path.isfile(HERMES):
    print('✅ Hermes sudah terinstall.')
else:
    print('⏳ Install Hermes Agent... (~5 menit)')
    !/usr/local/bin/uv pip install -e ".[all]" --python .venv/bin/python
    print('✅ Selesai.' if os.path.isfile(HERMES) else '❌ Gagal — cek error di atas.')

## 2️⃣ Konfigurasi
*Jalankan setiap kali runtime restart.*

In [ ]:
import os, subprocess, re, time

HERMES = '/content/hermes-agent/.venv/bin/hermes'
os.environ['PATH'] = '/content/hermes-agent/.venv/bin:' + os.environ.get('PATH', '')
os.environ['HERMES_HOME'] = os.path.expanduser('~/.hermes')

r = subprocess.run([HERMES, '--version'], capture_output=True, text=True)
print('✅' if r.returncode == 0 else '❌', r.stdout.strip() or r.stderr.strip())

In [ ]:
gemini_key = None

try:
    from google.colab import userdata
    gemini_key = userdata.get('GEMINI_API_KEY')
    print('✅ Key dari Colab Secrets.')
except Exception:
    pass

if not gemini_key:
    gemini_key = getpass.getpass('🔑 Masukkan Gemini API key: ')

gemini_key = re.sub(r'[\s]+', '', gemini_key)
os.environ['GEMINI_API_KEY'] = gemini_key
os.environ['GOOGLE_API_KEY'] = gemini_key
print(f'✅ Key siap ({len(gemini_key)} char).')

In [ ]:
subprocess.run([HERMES, 'config', 'set', 'GEMINI_API_KEY', gemini_key],
               capture_output=True, text=True, env=os.environ)
subprocess.run([HERMES, 'config', 'set', 'model', 'gemini/gemini-2.5-flash'],
               capture_output=True, text=True, env=os.environ)
print('✅ Model: gemini/gemini-2.5-flash')
print('💡 Ganti ke gemini-2.5-pro kalau mau lebih kuat (lebih lambat).')

In [ ]:
!{HERMES} doctor

> ⚠️ Warning soal symlink & setup adalah **normal** di Colab — bisa diabaikan.
> Yang penting: chat di bawah berhasil.

## 3️⃣ Chat

In [ ]:
def tanya(prompt, timeout=300):
    """Kirim prompt ke Hermes, tampilkan jawaban."""
    print(f'👤 {prompt}')
    print('🤖 ', end='', flush=True)
    r = subprocess.run([HERMES, '-z', prompt],
                       capture_output=True, text=True, timeout=timeout, env=os.environ)
    if r.returncode != 0:
        print('\n❌ Error:')
        print(r.stderr[-2000:])
    else:
        print(r.stdout)

print('✅ Fungsi tanya() siap.')

In [ ]:
tanya('Halo! Perkenalkan dirimu dan sebutkan tools yang aktif.')

In [ ]:
# ── Ganti pertanyaan di bawah, jalankan ulang sel ini kapan pun ──
tanya('Jelaskan manfaat AI dalam pendidikan dalam 3 poin singkat.')

In [ ]:
# ── Lanjutkan sesi chat sebelumnya ──
r = subprocess.run([HERMES, '--continue', '-z', 'Bisakah kamu elaborasi poin pertama?'],
                   capture_output=True, text=True, timeout=300, env=os.environ)
print('🤖', r.stdout or r.stderr)

## 4️⃣ Web UI — Dashboard via Cloudflare Tunnel

> ⚠️ `--insecure` diperlukan supaya tunnel bisa mengakses dashboard.
> URL-nya acak & sulit ditebak, tapi **matikan setelah selesai** (Sel terakhir).

In [ ]:
if os.path.isfile('/usr/local/bin/cloudflared'):
    print('✅ cloudflared sudah ada.')
else:
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
        -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
    print('✅ cloudflared terpasang.')

In [ ]:
!{HERMES} dashboard --stop 2>/dev/null; time.sleep(3)
print('✅ Dashboard lama dihentikan.')

In [ ]:
dash = subprocess.Popen(
    [HERMES, 'dashboard', '--host', '0.0.0.0', '--insecure'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ, 'BROWSER': 'none'}
)
time.sleep(8)

import urllib.request
try:
    urllib.request.urlopen('http://localhost:9119', timeout=5)
    print('✅ Dashboard menyala di port 9119.')
except Exception as e:
    print('⚠️ Belum siap:', e)

In [ ]:
get_ipython().system_raw('cloudflared tunnel --url http://localhost:9119 > /tmp/cf.log 2>&1 &')
time.sleep(10)

r = subprocess.run(['grep', '-o', 'https://[a-zA-Z0-9.-]*trycloudflare.com', '/tmp/cf.log'],
                   capture_output=True, text=True)
url = r.stdout.strip()

if url:
    print(f'🌐 Buka di browser:\n   {url}')
else:
    print('❌ URL belum muncul, jalankan ulang sel ini.')
    !tail -10 /tmp/cf.log

In [ ]:
# ── MATIKAN setelah selesai ──
!{HERMES} dashboard --stop
!pkill -f 'cloudflared tunnel' 2>/dev/null
print('✅ Dashboard & tunnel dimatikan.')

---
## 📋 Referensi Cepat

| Perintah | Cara |
|----------|------|
| Chat | `tanya('pertanyaan')` |
| Lanjut sesi | Sel `--continue` di atas |
| Daftar sesi | `!{HERMES} sessions list` |
| Lihat memori | `!{HERMES} memory` |
| Daftar skills | `!{HERMES} skills list` |
| Status | `!{HERMES} status` |
| Log | `!{HERMES} logs` |
| Update | `!{HERMES} update` |

📖 https://hermes-agent.nousresearch.com/docs  
🐛 https://github.com/NousResearch/hermes-agent